In [1]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# -*- coding: utf-8 -*-
from __future__ import annotations
import pandas as pd
from typing import Optional, Union, Sequence, Tuple
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────────────────────────────────────────

def make_engine(db_info: dict):
    url = (
        "mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
        "?charset=utf8mb4"
    ).format(**db_info)
    return create_engine(url, pool_recycle=3600, pool_pre_ping=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1) 특정 forecast_date로 유니크 티커 조회
#   - forecast_date가 None → forecast_date IS NULL 조건
#   - forecast_date가 'YYYY-MM-DD' 나 'YYYY-MM-DD HH:MM:SS' → 해당 일자/시각 매칭
#   - '같은 날'로 묶고 싶다면 use_date_only=True 로 DATE(forecast_date)=DATE(:dt)
# ─────────────────────────────────────────────────────────────────────────────
def get_unique_tickers_by_forecast_date(
    db_info: dict,
    forecast_date: Optional[str] = None,
    table_name: str = "valuation_forecast_result",
    use_date_only: bool = True,
) -> pd.Series:
    """
    Return: pd.Series of unique tickers (name='ticker')
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        if forecast_date is None:
            sql = text(f"""
                SELECT DISTINCT ticker
                FROM {table_name}
                WHERE forecast_date IS NULL
                ORDER BY ticker
            """)
            df = pd.read_sql(sql, conn)
        else:
            if use_date_only:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE DATE(forecast_date) = DATE(:dt)
                    ORDER BY ticker
                """)
            else:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE forecast_date = :dt
                    ORDER BY ticker
                """)
            df = pd.read_sql(sql, conn, params={"dt": forecast_date})

    return df["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# 2) indicator별 date1→date2 변화율 계산
#   - indicator: str 또는 [str, ...]  (None/'all'은 전체)
#   - 변화율 = (v2 / v1 - 1).  v1=0 또는 결측은 안전하게 제외
#   - 결과 정렬: pct_change(%) 내림차순
# Columns:
#   ['indicator','ticker','date1','date2','value_date1','value_date2',
#    'abs_change','pct_change']
# ─────────────────────────────────────────────────────────────────────────────
def get_indicator_change_rates(
    db_info: dict,
    forecast_date: Optional[str],
    indicator: Optional[Union[str, Sequence[str]]] = None,
    date1: str = "2027-03-31",
    date2: str = "2027-06-30",
    table_name: str = "valuation_forecast_result",
    use_date_only_for_forecast: bool = True,
    drop_zero_base: bool = True,
) -> pd.DataFrame:
    # ... (위쪽 동일: indicator_list 정리, where 조건 빌드) ...
    engine = make_engine(db_info)
    with engine.connect() as conn:
        conds = []
        params = {}

        # forecast_date filter
        if forecast_date is None:
            conds.append("forecast_date IS NULL")
        else:
            if use_date_only_for_forecast:
                conds.append("DATE(forecast_date) = DATE(:fdt)")
            else:
                conds.append("forecast_date = :fdt")
            params["fdt"] = forecast_date

        # indicator filter
        if indicator is None or (isinstance(indicator, str) and indicator.lower() == "all"):
            pass
        else:
            if isinstance(indicator, str):
                indicator_list = [indicator]
            else:
                indicator_list = list(indicator)
            placeholders = []
            for i, it in enumerate(indicator_list):
                key = f"i{i}"
                params[key] = it
                placeholders.append(f":{key}")
            conds.append(f"indicator IN ({', '.join(placeholders)})")

        # date filter
        conds.append("DATE(date) IN (DATE(:d1), DATE(:d2))")
        params["d1"] = date1
        params["d2"] = date2

        where_sql = " AND ".join(conds)

        # ⭐️ value가 문자일 수 있어 REPLACE 후 CAST: '1,234.56' -> 1234.56
        sql = text(f"""
            SELECT
                DATE(date) AS d,
                ticker,
                indicator,
                CAST(REPLACE(value, ',', '') AS DECIMAL(38, 8)) AS value
            FROM {table_name}
            WHERE {where_sql}
        """)
        raw = pd.read_sql(sql, conn, params=params)

    if raw.empty:
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2",
            "value_date1","value_date2","abs_change","pct_change"
        ])

    # pivot
    piv = (
        raw
        .assign(d=lambda x: pd.to_datetime(x["d"]).dt.date)
        .pivot_table(index=["indicator","ticker"], columns="d", values="value", aggfunc="last")
        .reset_index()
    )

    d1 = pd.to_datetime(date1).date()
    d2 = pd.to_datetime(date2).date()
    if d1 not in piv.columns: piv[d1] = pd.NA
    if d2 not in piv.columns: piv[d2] = pd.NA

    piv = piv.rename(columns={d1: "value_date1", d2: "value_date2"})
    out = piv[["indicator","ticker","value_date1","value_date2"]].copy()

    # ⭐️ 혹시 남아있는 문자열/공백/퍼센트 대비 2차 방어
    def _to_numeric_safe(s: pd.Series) -> pd.Series:
        # 문자열이면 콤마, % 제거 후 숫자 변환
        if s.dtype == "object":
            s = s.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False).str.strip()
        return pd.to_numeric(s, errors="coerce")

    out["value_date1"] = _to_numeric_safe(out["value_date1"])
    out["value_date2"] = _to_numeric_safe(out["value_date2"])

    # 결측/분모 0 제거
    out = out.dropna(subset=["value_date1","value_date2"])
    if drop_zero_base:
        out = out[out["value_date1"] != 0]

    # 변화/변화율
    out["abs_change"] = out["value_date2"] - out["value_date1"]
    out["pct_change"] = (out["value_date2"] / out["value_date1"] - 1.0) * 100.0

    # 메타/정렬
    out.insert(2, "date1", d1)
    out.insert(3, "date2", d2)
    out = out.sort_values(["indicator","pct_change"], ascending=[True, False]).reset_index(drop=True)

    return out



# ─────────────────────────────────────────────────────────────────────────────
# 사용 예시
# ─────────────────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     db_info = {
#         "host": "127.0.0.1",
#         "port": 3307,
#         "user": "stox7412",
#         "password": "*****",
#         "database": "investar",
#     }
#
#     # 1) 특정 forecast_date의 유니크 티커
#     #    NULL 날짜 기준이라면 forecast_date=None
#     tickers = get_unique_tickers_by_forecast_date(
#         db_info=db_info,
#         forecast_date=None,  # 예: NULL rows
#         table_name="valuation_forecast_result",
#         use_date_only=True,
#     )
#     print("[Unique tickers]\n", tickers.head())
#
#     # 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
#     df_change = get_indicator_change_rates(
#         db_info=db_info,
#         forecast_date=None,                    # 또는 '2027-06-30' 등
#         indicator=None,                        # 'all' 또는 ['sarima_forecast','lstm_forecast'] 등
#         date1="2027-03-31",
#         date2="2027-06-30",
#         table_name="valuation_forecast_result",
#         use_date_only_for_forecast=True,
#         drop_zero_base=True,
#     )
#     print("\n[Change table]\n", df_change.head(20))



# ── 사용 예시 ─────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # 1) forecast_date 목록
#     print(get_unique_forecast_dates().tail())
#
#     # 2) 키워드로 조회 (롱/와이드)
#     ex_long = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=False)
#     ex_wide = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=True)
#     print(ex_long.head())
#     print(ex_wide.head())
#
#     # 3) indicator 유니크
#     print(get_unique_indicators().head())
#     # 특정 키워드만
#     print(get_unique_indicators(keyword="forecast").head())
#
#     # 4) 두 forecast_date 비교
#     comp = compare_indicator_between_dates(
#         ticker="A005930",
#         indicator="revenue_ensemble_forecast",   # 예: 정확한 indicator 이름 입력
#         forecast_date_1="2025-10-26",
#         forecast_date_2="2025-10-29"
#     )
#     print(comp.tail())



In [2]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
2   2025-10-30
3   2025-10-31
4   2025-11-09
Name: forecast_date, dtype: datetime64[ns]


In [69]:
TICKER = "A131290"
FORECAST_DATE = "2025-11-09"

ex_long = get_series_by_keyword(ticker=TICKER , forecast_date=FORECAST_DATE, keyword="rev", wide=False)

In [70]:
ex_long.tail(15)

,date,ticker,indicator,value,forecast_date
843,2026-06-30,A131290,revenue_prophet_ttm,3.917621e+08,2025-11-09
844,2026-06-30,A131290,revenue_lstm_ttm,3.289232e+08,2025-11-09
845,2026-06-30,A131290,revenue_theta_ttm,4.522003e+08,2025-11-09
846,2026-09-30,A131290,revenue_sarima,1.734649e+08,2025-11-09
847,2026-09-30,A131290,revenue_sarima_exog,1.974335e+08,2025-11-09
848,2026-09-30,A131290,revenue_ets,1.347814e+08,2025-11-09
849,2026-09-30,A131290,revenue_prophet,1.019839e+08,2025-11-09
850,2026-09-30,A131290,revenue_lstm,8.575551e+07,2025-11-09
851,2026-09-30,A131290,revenue_theta,1.255493e+08,2025-11-09
852,2026-09-30,A131290,revenue_sarima_ttm,5.829439e+08,2025-11-09


In [71]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker  revenue_ets  revenue_ets_ttm  revenue_lstm  \
0 2004-12-31  A131290          0.0              NaN           0.0   
1 2005-12-31  A131290          0.0              NaN           0.0   
2 2006-12-31  A131290          0.0              NaN           0.0   
3 2007-12-31  A131290          0.0              0.0           0.0   
4 2008-12-31  A131290          0.0              0.0           0.0   

   revenue_lstm_ttm  revenue_prophet  revenue_prophet_ttm  revenue_sarima  \
0               NaN              0.0                  NaN             0.0   
1               NaN              0.0                  NaN             0.0   
2               NaN              0.0                  NaN             0.0   
3               0.0              0.0                  0.0             0.0   
4               0.0              0.0                  0.0             0.0   

   revenue_sarima_exog  revenue_sarima_exog_ttm  revenue_sarima_ttm  \
0                  0.0                      NaN    

In [72]:
ex_pivot.columns.tolist()

['date',
 'ticker',
 'revenue_ets',
 'revenue_ets_ttm',
 'revenue_lstm',
 'revenue_lstm_ttm',
 'revenue_prophet',
 'revenue_prophet_ttm',
 'revenue_sarima',
 'revenue_sarima_exog',
 'revenue_sarima_exog_ttm',
 'revenue_sarima_ttm',
 'revenue_theta',
 'revenue_theta_ttm']

In [73]:
ex_pivot[['date', 'revenue_ets', 'revenue_prophet', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_theta']].tail(10)

,date,revenue_ets,revenue_prophet,revenue_sarima,revenue_sarima_exog,revenue_theta
63,2024-06-30,7.604882e+07,7.604882e+07,7.604882e+07,7.604882e+07,7.604882e+07
64,2024-09-30,1.106759e+08,1.106759e+08,1.106759e+08,1.106759e+08,1.106759e+08
65,2024-12-31,1.031176e+08,1.031176e+08,1.031176e+08,1.031176e+08,1.031176e+08
66,2025-03-31,8.303146e+07,8.303146e+07,8.303146e+07,8.303146e+07,8.303146e+07
67,2025-06-30,1.176139e+08,1.176139e+08,1.176139e+08,1.176139e+08,1.176139e+08
68,2025-09-30,1.258462e+08,9.551803e+07,1.402246e+08,1.555271e+08,1.220416e+08
69,2025-12-31,1.164918e+08,9.714778e+07,1.393132e+08,1.355502e+08,1.159213e+08
70,2026-03-31,1.010367e+08,9.874211e+07,1.243319e+08,6.161133e+07,1.003556e+08
71,2026-06-30,1.193189e+08,1.003541e+08,1.458339e+08,1.227843e+08,1.138818e+08
72,2026-09-30,1.347814e+08,1.019839e+08,1.734649e+08,1.974335e+08,1.255493e+08


In [74]:
psr_long = get_series_by_keyword(ticker=TICKER, forecast_date= FORECAST_DATE, keyword="psr", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker       psr  psr_ETS  psr_LSTM  psr_Prophet  \
0 2015-05-31  A131290  1.670988      NaN       NaN          NaN   
1 2015-06-30  A131290  1.425398      NaN       NaN          NaN   
2 2015-07-31  A131290  1.183269      NaN       NaN          NaN   
3 2015-08-31  A131290  1.413862      NaN       NaN          NaN   
4 2015-09-30  A131290  1.348606      NaN       NaN          NaN   

   psr_SARIMA_exog  psr_SARIMA_noexog  psr_Theta  
0              NaN                NaN        NaN  
1              NaN                NaN        NaN  
2              NaN                NaN        NaN  
3              NaN                NaN        NaN  
4              NaN                NaN        NaN  


In [75]:
psr_pivot.tail(20)

,date,ticker,psr,psr_ETS,psr_LSTM,psr_Prophet,psr_SARIMA_exog,psr_SARIMA_noexog,psr_Theta
120,2025-05-31,A131290,1.483180,NaN,NaN,NaN,NaN,NaN,NaN
121,2025-06-30,A131290,1.471999,NaN,NaN,NaN,NaN,NaN,NaN
122,2025-07-31,A131290,1.453368,NaN,NaN,NaN,NaN,NaN,NaN
123,2025-08-31,A131290,1.367372,NaN,NaN,NaN,NaN,NaN,NaN
124,2025-09-30,A131290,1.940901,NaN,NaN,NaN,NaN,NaN,NaN
125,2025-10-31,A131290,1.889921,NaN,NaN,NaN,NaN,NaN,NaN
126,2025-11-30,A131290,1.919052,NaN,NaN,NaN,NaN,NaN,NaN
127,2025-12-31,A131290,NaN,2.097501,1.690776,2.842225,2.126575,1.907624,2.142961
128,2026-01-31,A131290,NaN,2.083264,1.755577,2.809858,2.110641,1.907624,2.175255
129,2026-02-28,A131290,NaN,1.896754,1.828847,2.550771,1.933417,1.907624,1.951413


In [76]:
mc_long = get_series_by_keyword(ticker=TICKER, forecast_date=FORECAST_DATE, keyword="mc_", wide=False)

# ex_long → indicator를 컬럼으로 피벗
mc_pivot = (
    mc_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
mc_pivot.columns.name = None

# 확인
print(mc_pivot.head())

        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-12-31  A131290  9.291582e+08  6.131787e+08  1.117879e+09   
1 2026-01-31  A131290  9.228514e+08  6.366798e+08  1.105149e+09   
2 2026-02-28  A131290  8.402308e+08  6.632519e+08  1.003247e+09   
3 2026-03-31  A131290  9.019043e+08  6.822331e+08  1.094144e+09   
4 2026-04-30  A131290  9.945831e+08  7.012167e+08  1.220894e+09   

   mc_sarima_exog  mc_sarima_noexog      mc_theta  
0    1.045686e+09      9.160091e+08  9.399204e+08  
1    1.037850e+09      9.160091e+08  9.540851e+08  
2    9.507051e+08      9.160091e+08  8.559060e+08  
3    9.329692e+08      9.947948e+08  9.098817e+08  
4    1.027540e+09      9.947948e+08  9.899661e+08  


In [77]:
mc_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_exog,mc_sarima_noexog,mc_theta
0,2025-12-31,A131290,9.291582e+08,6.131787e+08,1.117879e+09,1.045686e+09,9.160091e+08,9.399204e+08
1,2026-01-31,A131290,9.228514e+08,6.366798e+08,1.105149e+09,1.037850e+09,9.160091e+08,9.540851e+08
2,2026-02-28,A131290,8.402308e+08,6.632519e+08,1.003247e+09,9.507051e+08,9.160091e+08,8.559060e+08
3,2026-03-31,A131290,9.019043e+08,6.822331e+08,1.094144e+09,9.329692e+08,9.947948e+08,9.098817e+08
4,2026-04-30,A131290,9.945831e+08,7.012167e+08,1.220894e+09,1.027540e+09,9.947948e+08,9.899661e+08
5,2026-05-31,A131290,1.002419e+09,7.091292e+08,1.225808e+09,1.033455e+09,9.947948e+08,1.017051e+09
6,2026-06-30,A131290,9.822109e+08,6.462659e+08,1.149710e+09,1.033270e+09,1.048628e+09,1.015218e+09
7,2026-07-31,A131290,1.028406e+09,6.484285e+08,1.203202e+09,1.078775e+09,1.048628e+09,1.070854e+09
8,2026-08-31,A131290,9.724809e+08,6.523850e+08,1.110149e+09,1.024364e+09,1.048628e+09,1.007905e+09
9,2026-09-30,A131290,9.707402e+08,6.661768e+08,1.111373e+09,1.090316e+09,1.112038e+09,9.597954e+08


In [23]:
tickers = get_unique_tickers_by_forecast_date(
    db_info=db_info,
    forecast_date=None,  # 예: NULL rows
    table_name="Korea_company_valuation_ver2",
    use_date_only=True,
)
print("[Unique tickers]\n", tickers.head())

[Unique tickers]
 0    A000270
1    A000500
2    A000660
3    A001440
4    A002350
Name: ticker, dtype: object


In [24]:
tickers

0      A000270
1      A000500
2      A000660
3      A001440
4      A002350
5      A004000
6      A005380
7      A005930
8      A006400
9      A006910
10     A007700
11     A009150
12     A010120
13     A010140
14     A011780
15     A031980
16     A033500
17     A035420
18     A035720
19     A036190
20     A042370
21     A042660
22    A042700 
23     A043150
24     A044820
25     A051910
26     A059090
27     A060980
28     A068270
29     A071280
30     A071970
31     A073240
32     A077360
33     A082740
34     A084370
35     A086390
36     A093520
37     A095610
38     A103140
39     A103590
40     A105630
41     A114810
42     A123330
43     A123700
44     A131290
45     A140860
46     A161390
47     A207940
48     A214150
49     A232140
50     A253590
51     A375500
Name: ticker, dtype: object

In [89]:
# 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date= "2025-10-31",                    # 또는 '2027-06-30' 등
    indicator= 'mc_sarima_exog',                        # 'all' 또는 ['sarima_forecast','lstm_forecast'] 등
    date1="2025-11-30",
    date2="2026-11-30",
    table_name="Korea_company_valuation_ver2",
    use_date_only_for_forecast=True,
    drop_zero_base=True,
)
print("\n[Change table]\n", df_change.tail(20))


[Change table]
 d        indicator   ticker       date1       date2   value_date1  \
10  mc_sarima_exog  A131970  2025-11-30  2026-11-30  1.339920e+09   
11  mc_sarima_exog  A010120  2025-11-30  2026-11-30  2.081106e+10   
12  mc_sarima_exog  A114810  2025-11-30  2026-11-30  5.131411e+08   
13  mc_sarima_exog  A044820  2025-11-30  2026-11-30  1.665905e+08   
14  mc_sarima_exog  A131290  2025-11-30  2026-11-30  8.339153e+08   
15  mc_sarima_exog  A095610  2025-11-30  2026-11-30  1.108228e+09   
16  mc_sarima_exog  A009150  2025-11-30  2026-11-30  2.215294e+10   
17  mc_sarima_exog  A005930  2025-11-30  2026-11-30  8.545471e+11   
18  mc_sarima_exog  A375500  2025-11-30  2026-11-30  2.497654e+09   
19  mc_sarima_exog  A005380  2025-11-30  2026-11-30  7.188432e+10   
20  mc_sarima_exog  A000270  2025-11-30  2026-11-30  5.885667e+10   
21  mc_sarima_exog  A006400  2025-11-30  2026-11-30  3.419951e+10   
22  mc_sarima_exog  A042370  2025-11-30  2026-11-30  2.925160e+08   
23  mc_sarima_exo